# Spam Email Classifier — Machine Learning & NLP
## MCA Major/Minor Project Implementation

### Project Objective
The primary objective of this project is to construct, train, and evaluate a Natural Language Processing (NLP) system capable of classifying email and SMS communications as either **Spam** or **Not Spam (Ham)**.

### Workflow
```
Email Dataset → Data Cleaning → Text Preprocessing → TF-IDF Vectorization → Train/Test Split → Model Training (Naive Bayes vs. SVM) → Comparative Evaluation
```

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report, roc_auc_score
)

# Add project root to path
sys.path.append(os.path.abspath('..'))
from src.preprocessing import clean_text

sns.set_theme(style="whitegrid")
%matplotlib inline
print("Libraries successfully imported!")

## 1. Exploratory Data Analysis (EDA)
Load the curated dataset and analyze class distribution.

In [ ]:
df = pd.read_csv('../dataset/spam.csv')
print("Dataset Shape:", df.shape)
df.head()

In [ ]:
# Class distribution
class_counts = df['label'].value_counts()
print(class_counts)

plt.figure(figsize=(6, 4))
sns.barplot(x=class_counts.index, y=class_counts.values, palette=["#22c55e", "#ef4444"])
plt.title("Class Distribution (Ham vs. Spam)")
plt.xlabel("Class")
plt.ylabel("Count")
plt.show()

## 2. Text Preprocessing
Normalize URLs, email addresses, and currency symbols, lowercase tokens, remove punctuation, numbers, and stopwords.

In [ ]:
# Example of preprocessing transformation
sample_email = "Congratulations! You won $5,000 cash. Click https://prize-claim.com now!"
print("Original:", sample_email)
print("Cleaned :", clean_text(sample_email))

# Apply to dataset
df['cleaned_text'] = df['text'].apply(clean_text)
df['label_binary'] = df['label'].map({'ham': 0, 'spam': 1})
df = df[df['cleaned_text'].str.strip() != ''].reset_index(drop=True)
print(f"Processed {len(df)} samples.")

## 3. Train / Test Split & Feature Extraction (TF-IDF)
We split the dataset 80/20 with stratification to preserve the ham-to-spam ratio.

In [ ]:
X = df['cleaned_text']
y = df['label_binary']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

vectorizer = TfidfVectorizer(ngram_range=(1, 2), max_features=6000, sublinear_tf=True)
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

print("TF-IDF Train Matrix Shape:", X_train_tfidf.shape)
print("TF-IDF Test Matrix Shape:", X_test_tfidf.shape)

## 4. Model Training: Naive Bayes vs. Support Vector Machine
- **Multinomial Naive Bayes**: Uses Bayes' Theorem under conditional feature independence assumption.
- **Support Vector Machine**: Maximizes the margin between separating hyperplane and support vectors.

In [ ]:
# 1. Multinomial Naive Bayes
nb_model = MultinomialNB(alpha=0.1)
nb_model.fit(X_train_tfidf, y_train)

# 2. Linear SVM with CalibratedClassifierCV
base_svm = LinearSVC(C=1.0, class_weight='balanced', random_state=42, max_iter=3000)
svm_model = CalibratedClassifierCV(estimator=base_svm, method='sigmoid', cv=3)
svm_model.fit(X_train_tfidf, y_train)

print("Both models trained successfully!")

## 5. Comparative Evaluation
Evaluate Accuracy, Precision, Recall, F1-Score, and plot Confusion Matrices.

In [ ]:
models = {
    "Multinomial Naive Bayes": nb_model,
    "Support Vector Machine": svm_model
}

results = []

for name, model in models.items():
    y_pred = model.predict(X_test_tfidf)
    y_prob = model.predict_proba(X_test_tfidf)[:, 1]
    
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, pos_label=1)
    rec = recall_score(y_test, y_pred, pos_label=1)
    f1 = f1_score(y_test, y_pred, pos_label=1)
    auc = roc_auc_score(y_test, y_prob)
    
    results.append({
        "Model": name,
        "Accuracy": f"{acc*100:.2f}%",
        "Precision (Spam)": f"{prec*100:.2f}%",
        "Recall (Spam)": f"{rec*100:.2f}%",
        "F1-Score (Spam)": f"{f1*100:.2f}%",
        "ROC-AUC": f"{auc:.4f}"
    })

pd.DataFrame(results)

In [ ]:
# Confusion Matrices
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

for ax, (name, model) in zip(axes, models.items()):
    y_pred = model.predict(X_test_tfidf)
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False, ax=ax,
                xticklabels=["Pred Ham", "Pred Spam"], yticklabels=["Actual Ham", "Actual Spam"])
    ax.set_title(f"{name} Confusion Matrix")

plt.tight_layout()
plt.show()

## 6. Live Inference on Custom Unseen Emails

In [ ]:
test_emails = [
    "URGENT: Click here http://bank-update.ru to claim your $1,000 lottery winnings now!",
    "Hi Team, please find the quarterly financial budget summary attached for tomorrow's meeting.",
    "Claim your free gift card before the midnight deadline! Text WIN to 89342"
]

for email in test_emails:
    cleaned = clean_text(email)
    vec = vectorizer.transform([cleaned])
    pred_nb = nb_model.predict(vec)[0]
    prob_nb = nb_model.predict_proba(vec)[0][1]
    
    print("-" * 60)
    print("Text:", email)
    print(f"Prediction: {'SPAM' if pred_nb == 1 else 'HAM'} (Confidence: {prob_nb*100:.1f}%)")